# RG3 가스 실제분류기1~3개 축소 비교
한시드·한설정으로재학습. 원본RF2기준재현, 안쪽선택조합과사후개발상위를구분한다.


In [1]:
from pathlib import Path
import itertools,json,nbformat
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import NearestNeighbors
ROOT=Path.cwd();OUT=ROOT/"reports/tables"
# Existing RG3 baseline helper, preserving its2RF reference behavior.
n=nbformat.read("notebooks/22_rg3_nested_signal_ensemble.ipynb",as_version=4)
s="\n".join(c.source for c in n.cells if c.cell_type=="code")
exec(s[:s.index("scores=[];thresholds=[];foldlogs=[]")])
d=d.reset_index(drop=True);groups=d.comparison_block;blocks=groups.unique().tolist()
# Original physical feature definitions; no imported50-head bundle fit.
n=nbformat.read("notebooks/154_rg3_gas_model_export.ipynb",as_version=4)
definition=n.cells[1].source
exec(definition[definition.index("def physical("):definition.index("def fit_conditional(")])
KEYS=["EQUIP_CD","product","TimeStamp"]
names=["RF_std4","ET_thermal6","LR_thermal6","LR_pressure6"]
variants=[v for k in [1,2,3] for v in itertools.combinations(names,k)]
localmemo={}
def local(ref,q,kind):
 key=(tuple(ref),tuple(q),kind)
 if key in localmemo:return localmemo[key]
 normal=ref[y[ref]==0]
 control=["Barrel_Temperature_5","Max_Injection_Speed","Filling_Time","Hopper_Temperature"] if kind=="thermal" else ["Max_Injection_Speed","Filling_Time","Barrel_Temperature_5","Mold_Temperature_3"]
 ri=SimpleImputer(keep_empty_features=True).fit(physical(d.iloc[normal],kind))
 ci=SimpleImputer(keep_empty_features=True).fit(d.iloc[normal][control])
 cs=RobustScaler().fit(ci.transform(d.iloc[normal][control]))
 raw=ri.transform(physical(d.iloc[normal],kind))
 peer={"name":kind,"controls":control,"raw_imp":ri,"control_imp":ci,"control_sc":cs,"raw":raw,"keys":d.iloc[normal][KEYS].reset_index(drop=True),
 "floor":np.maximum(raw.std(axis=0)*.1,1e-6),"nn":NearestNeighbors(n_neighbors=min(31,len(normal))).fit(cs.transform(ci.transform(d.iloc[normal][control])))}
 out=conditional_features(peer,d.iloc[q].reset_index(drop=True))
 localmemo[key]=out;return out
memo={};audits=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 audits.append({"train":len(ref),"train_gas":int(y[ref].sum()),"query":len(q),"timestamp_overlap":len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q])),
 "shared_comparison_block":len(set(groups.iloc[ref])&set(groups.iloc[q]))})
 pair=pair_predict(ref,q)
 out={"RF_std4":pair[:,0],"baseline_RF2_max":pair.max(axis=1)}
 zq={}
 for kind in ["thermal","pressure"]:
  tr=local(ref,ref,kind);te=local(ref,q,kind);zq[kind]=te
  imp=SimpleImputer(keep_empty_features=True).fit(tr)
  a=imp.transform(tr);b=imp.transform(te)
  if kind=="thermal":
   m=ExtraTreesClassifier(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2).fit(a,y[ref])
   out["ET_thermal6"]=m.predict_proba(b)[:,1]
  sc=RobustScaler().fit(a[y[ref]==0])
  m=LogisticRegression(C=.1,class_weight="balanced",max_iter=2000,random_state=42).fit(sc.transform(a),y[ref])
  out["LR_"+kind+"6"]=m.predict_proba(sc.transform(b))[:,1]
 memo[key]=(out,zq)
 return memo[key]
def stats(yy,pr):
 tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
 return {"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),"F1":float(2*tp/(2*tp+fp+fn)) if tp else 0,
 "precision":float(tp/(tp+fp)) if tp+fp else 0,"recall":float(tp/(tp+fn)) if tp+fn else 0}
def calibrated(inner,zinner,outer,zouter,ref,q):
 ip={};op={};cuts={}
 for name in names+["baseline_RF2_max"]:
  quant=.975 if "thermal" in name else .99
  cut=float(np.quantile(inner[name][y[ref]==0],quant,method="higher"))
  ip[name]=inner[name]>cut;op[name]=outer[name]>cut;cuts[name]={"score":cut}
  if "thermal" in name:
   gate=float(np.quantile(zinner["thermal"][y[ref]==0,4],.05))
   ip[name]&=zinner["thermal"][:,4]<gate
   op[name]&=zouter["thermal"][:,4]<gate
   cuts[name]["thermal_gate"]=gate
  if name=="LR_pressure6":
   def direction(ids):
    a=d.iloc[ids]
    return ((a.Max_Injection_Pressure__delta1.round(2)<0)&(a.Max_Injection_Speed__delta1.round(2)>0)&(a.Max_Switch_Over_Pressure__delta1.round(2)<a.Max_Injection_Pressure__delta1.round(2))).to_numpy()
   ip[name]&=direction(ref);op[name]&=direction(q)
 return ip,op,cuts
records=[];choice_rows=[];component_rows=[]
for held in blocks:
 q=np.flatnonzero(groups.eq(held));ref=np.flatnonzero(~groups.eq(held))
 outer,zo=predict(ref,q)
 inner={n:np.full(len(ref),np.nan) for n in outer}
 zi={k:np.full((len(ref),6),np.nan) for k in ["thermal","pressure"]}
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy()
  scores,z=predict(ref[~mask],ref[mask])
  for name in inner:inner[name][mask]=scores[name]
  for kind in zi:zi[kind][mask]=z[kind]
 assert all(np.isfinite(v).all() for v in inner.values())
 ip,op,cuts=calibrated(inner,zi,outer,zo,ref,q)
 options=[]
 for combo in variants:
  label="+".join(combo)
  pr=np.any(np.column_stack([ip[n] for n in combo]),axis=1)
  met=stats(y[ref],pr)
  if met["FP"]<=int(np.floor((y[ref]==0).sum()*.03)):
   options.append((met["F1"],met["TP"],-met["FP"],-len(combo),label))
  out=np.any(np.column_stack([op[n] for n in combo]),axis=1)
  for j,i in enumerate(q):records.append({"policy":label,"model_count":len(combo),"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"actual":int(y[i]),"prediction":int(out[j])})
 if not options:
  # Keep all evaluated shots, no guessed gas predictions if no candidate meets inner budget.
  selected="none";pr=np.zeros(len(q),bool);best=(0,0,0,0,"none")
 else:
  best=max(options);selected=best[-1];combo=selected.split("+")
  pr=np.any(np.column_stack([op[n] for n in combo]),axis=1)
 choice_rows.append({"heldout_block":held,"selected":selected,"model_count":0 if selected=="none" else len(combo),"inner_F1":best[0],"train_gas":int(y[ref].sum())})
 for j,i in enumerate(q):
  records.append({"policy":"primary_inner_selected_max3","model_count":0 if selected=="none" else len(combo),"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"actual":int(y[i]),"prediction":int(pr[j])})
  records.append({"policy":"reference_RF2_original","model_count":2,"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"actual":int(y[i]),"prediction":int(op["baseline_RF2_max"][j])})
  for name in names:component_rows.append({"head":name,"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"actual":int(y[i]),"score":float(outer[name][j]),"prediction":int(op[name][j]),"cutoff":cuts[name]["score"]})
 pd.DataFrame(choice_rows).to_csv(OUT/"rg3_187_small_classifier_progress.csv",index=False)
 print("Finished actual small-model refit",held,selected,flush=True)
pp=pd.DataFrame(records);metrics=[]
for policy,a in pp.groupby("policy",sort=False):
 assert len(a)==583 and a.TimeStamp.is_unique
 metrics.append({"policy":policy,"classifiers_min":int(a.model_count.min()),"classifiers_max":int(a.model_count.max()),"N":583,"gas":17,**stats(a.actual,a.prediction)})
mm=pd.DataFrame(metrics)
pp.to_csv(OUT/"rg3_187_small_classifier_predictions.csv",index=False)
mm.to_csv(OUT/"rg3_187_small_classifier_metrics.csv",index=False)
pd.DataFrame(component_rows).to_csv(OUT/"rg3_187_single_head_predictions.csv",index=False)
pd.DataFrame(choice_rows).to_csv(OUT/"rg3_187_inner_selected_models.csv",index=False)
pd.DataFrame(audits).to_csv(OUT/"rg3_187_small_classifier_fit_audit.csv",index=False)
assert pd.DataFrame(audits).timestamp_overlap.eq(0).all() and pd.DataFrame(audits).shared_comparison_block.eq(0).all()
# Confirm baseline reproduces its original protocol, not merely similar performance.
old=pd.read_csv(OUT/"rg3_nested_ensemble_predictions.csv",parse_dates=["TimeStamp"])
old=old[old.scheme.eq("leave_one_block_out")&old.model.eq("max_union")]
check=pp[pp.policy.eq("reference_RF2_original")].merge(old[["TimeStamp","predicted_train_fpr1"]],on="TimeStamp",validate="one_to_one")
assert len(check)==583 and check.prediction.eq(check.predicted_train_fpr1.astype(int)).all()
primary=mm[mm.policy.eq("primary_inner_selected_max3")]
top=mm[~mm.policy.isin(["primary_inner_selected_max3","reference_RF2_original"])].sort_values(["F1","FP","classifiers_max"],ascending=[False,True,True]).head(10)
best=top.iloc[0].policy
cases=pp[pp.policy.eq(best)].merge(d[["TimeStamp","Reason","comparison_block"]],on="TimeStamp",validate="one_to_one")
cases[cases.actual.eq(1)].to_csv(OUT/"rg3_187_best_fixed_all17_gas.csv",index=False)
print("\nPRIMARY\n",primary.to_string(index=False))
print("\nREFERENCE RF2\n",mm[mm.policy.eq("reference_RF2_original")].to_string(index=False))
print("\nFIXED DEVELOPMENT CANDIDATES\n",top.to_string(index=False))
print("\nALL17\n",cases[cases.actual.eq(1)][["TimeStamp","prediction","Reason"]].to_string(index=False))
lines=["# RG3 가스 모델수 축소: 실제1~3개 분류기 재학습","",
 "RG3 공식정상566/가스17=583샷. 이전50개와같은데이터/5개comparison block검증을사용한다.",
 "시드반복과C값반복은없앴다. RF흔들림4/ET열6/LR열6/LR압력6을각1개씩학습해서최대3개OR조합을비교한다.",
 "단독LR의C는.1고정, RF/ET는seed42고정이다. 열모델은기존학습정상기준.975/열변화하위.05확인,압력은.99/기존방향확인을유지한다.",
 "RF2기준은기존22번예측583개와정확히일치함을확인했다. 정상관계잔차회귀는RF2기준재현에만쓰고새1~3개후보에는쓰지않는다.",
 "주모델은안쪽구간검증F1및정상FP3%예산으로최대3개조합을선택한다.",
 "고정후보전체평가상위는사후개발비교이며독립성능이라고주장하지않는다.",
 "기존센서과거이력피처를재사용하며원본라벨값이피처로들어가지않는다. 같은기존평가protocol이고미관측새Run/미래최종독립검증은아니다.",
 "분류기수는최종조합의학습분류기수다. 비교실험의교차검증학습횟수/나무개수/정상참조최근접검색은분류기개수와다르다.","",
 "## 학습내부선택",primary.to_string(index=False),"",
 "## RF2기준",mm[mm.policy.eq("reference_RF2_original")].to_string(index=False),"",
 "## 고정후보개발비교",top.to_string(index=False),"",
 "## 개발최고후보의가스17개",cases[cases.actual.eq(1)][["TimeStamp","prediction","Reason"]].to_string(index=False)]
(ROOT/"reports/rg3_187_small_classifier_comparison.md").write_text("\n".join(lines),encoding="utf-8")


Finished actual small-model refit 20201021_001626_RG3 RF_std4+LR_thermal6


Finished actual small-model refit 20201022_002405_RG3 LR_pressure6


Finished actual small-model refit 20201022_044026_RG3 LR_thermal6


Finished actual small-model refit 20201023_010128_RG3 RF_std4+ET_thermal6+LR_thermal6


Finished actual small-model refit 20201023_044148_RG3 LR_pressure6



PRIMARY
                      policy  classifiers_min  classifiers_max   N  gas  TP  FP  FN  TN  F1  precision  recall
primary_inner_selected_max3                1                3 583   17   0   1  17 565 0.0        0.0     0.0

REFERENCE RF2
                 policy  classifiers_min  classifiers_max   N  gas  TP  FP  FN  TN       F1  precision   recall
reference_RF2_original                2                2 583   17   4   6  13 560 0.296296        0.4 0.235294

FIXED DEVELOPMENT CANDIDATES
                           policy  classifiers_min  classifiers_max   N  gas  TP  FP  FN  TN       F1  precision   recall
                         RF_std4                1                1 583   17   3  13  14 553 0.181818   0.187500 0.176471
            RF_std4+LR_pressure6                2                2 583   17   3  13  14 553 0.181818   0.187500 0.176471
             RF_std4+ET_thermal6                2                2 583   17   3  14  14 552 0.176471   0.176471 0.176471
             RF_s

3029